In [30]:
import pandas as pd
from tqdm import tqdm
import random
from transformers import AutoModelForCausalLM, AutoTokenizer

d:\miniconda\envs\tinkoff_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm

A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.1 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "C:\Users\palki\AppData\Roaming\Python\Python312\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "C:\Users\palki\Ap

### Читаем данные

In [16]:
# смотрим размеченные данные
data = pd.read_csv('data/marked_data.csv')
data.head(5)

,text,category
0,"Заказали 14.10.2017 , получили 25.10.2017 \r\n...",одежда
1,"футболка хорошего качества,но футболка не как ...",одежда
2,Все отлично!!!,нет товара
3,"Рисунок не очень чёткий, а ткань прозрачная, в...",текстиль
4,плохо!!!Низ рваный..деньги не вернули!Открыла ...,нет товара


In [ ]:
# распределение размеченных данных
print(data['category'].value_counts())

category
одежда                    893
нет товара                740
текстиль                  111
обувь                      37
украшения и аксессуары     22
товары для детей            5
бытовая техника             5
посуда                      3
электроника                 1
нет категории               1
Name: count, dtype: int64


### Анализ ошибок и корректировка

In [ ]:
# печать данных определенной категории
def print_category(category):
    for text in data[data['category'] == category]['text'].to_list():
        print(text)
        print()

In [ ]:
# после автоматической разметки в категорию <бытовая техника> попали все отзывы со словом <стирка> и его формы
print_category('бытовая техника')

заказ пришёл через 55 дней, ещё и с пятном еле еле отстирали ставлю 2

Деньги на ветер иначе не назовёшь! После стирки перекосилось! Цвета естественно облезли, размер маломерит как минимум на 2-3 размера! Продавец просил 5! Хотя по факту не за что совершенно!

Минус после стирку пошёл весь в катушки 

неприятный запах, после стирки не ушел, кнопка очень тугая, сразу отодралась, приклеена была на клей, нитки торчали везде, открыла спор, вернули 200 рублей

Это шикарно!! Пришло за 2 недели до Иркутска, запаха не было, стирки пережила хорошо. Боже,это потрясно!!! Спасибо большое!))) Очень советую)))))



In [ ]:
# после автоматической разметки в категорию <посуда> попали все отзывы про бюстгалтеры, из-за слова <чашка>
print_category('посуда')

пришел товар быстро, но абсолютно не тот размер чашки огромные  заказала 80 А

Ткань синтетика, но приятная, чашки маленькие. Доставка быстрая

заказ доставлен за 11 дней.
кружевное и красивое, на этом плюсы закончились.
очень длинное, наверно на 2х метровых девушек.край кружева на груди несимметричен и смотрится некрасиво.



In [ ]:
# явно неправильная разметка
print_category('электроника')

заказ шел долго. супер эффекта не нашла. стрелки пошли после первой носки! таких денег не стоят однозначно.



In [ ]:
# новое распределение данных
data = data[~data['category'].isin(['бытовая техника', 'электроника', 'нет категории'])]
data.loc[data['category'] == 'посуда', 'category'] = 'одежда'
print(data['category'].value_counts())

category
одежда                    896
нет товара                740
текстиль                  111
обувь                      37
украшения и аксессуары     22
товары для детей            5
Name: count, dtype: int64


### Генерация синтетических данных

In [ ]:
model_name = "Qwen/Qwen3-4B-Instruct-2507"

# загружаем модель и токенизатор
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
# категории для которых надо генерировать данные
categories = ['бытовая техника', 'обувь', 'посуда', 'текстиль', 'товары для детей', 'украшения и аксессуары', 'электроника']

# категории товаров с описанием
description = {
    'бытовая техника': 'холодильники, стиральные машины, плиты, микроволновки, чайники, пылесосы и другая техника для дома.',
    'обувь': 'кроссовки, ботинки, туфли, сандалии, сапоги и другая обувь для взрослых и детей.',
    'посуда': 'только тарелки, кружки, чашки для еды, столовые приборы и кухонная утварь; НЕ включает одежду, бельё или чашки бюстгальтеров.',
    'текстиль': 'только постельное бельё, покрывала, наволочки, кухонные полотенца, пледы, портьеры, шторы, ковры и так далее.',
    'товары для детей': 'игрушки, детская одежда, детская мебель, товары для ухода за детьми, коляски, автокресла.',
    'украшения и аксессуары': 'серьги, кольца, браслеты, ожерелья, ремни, шарфы, сумки, очки и другие модные аксессуары.',
    'электроника': 'смартфоны, планшеты, ноутбуки, камеры, наушники, колонки, смарт-часы и прочая электроника.',
}

# категории товаров с примерами
examples = {
    'бытовая техника': [
        'Заказала стиральную машину, пришла быстро, упаковка целая, но при включении гремит и сильно прыгает по полу.',
        'Пришёл с небольшой вмятиной, но работает без проблем. Доставка быстрая.',
        'Плита греет хорошо, но запах пластика не проходил неделю. Сейчас всё нормально.'
    ],
    'обувь': [
        'Купила новые кроссовки, очень удобные.',
        'Качество отличное, запаха нет, фурнитура надёжная. Брала мужу — доволен.',
        'Выглядят стильно, но ремешок короткий — на широкую ногу не подойдут.'
    ],
    'посуда': [
        'Тарелки пришли целые, хорошо упакованы. Очень красивые, рисунок яркий. Мыть можно в посудомойке — ничего не стерлось.',
        'Контейнеры хорошие, крышки плотно закрываются. Удобно брать еду с собой на работу.',
        'Ножи тупые, пришлось точить сразу. За эту цену ожидала лучшее качество.'
    ],
    'текстиль': [
        'За свою цену просто находка. Мягкие, но при этом плотные. Взяла набор, теперь думаю заказать ещё.',
        'Цвет вживую немного теплее, чем на фото, но в интерьере даже лучше смотрится. Материал качественный, не просвечивает.',
        'Купила постельное бельё из хлопка, качество хорошее.'
    ],
    'товары для детей': [
        'Игрушка пришла быстро, без запаха, ребёнок в восторге.',
        'Размер не соответсвует ,немного не тот цвет,рассчитан для девочек лет 10-12',
        'Тонкая синтетика, но за такие деньги норм, ребенок доволен'
    ],
    'украшения и аксессуары': [
        'у очков в белой оправе было деформировано стекло , все как будто "плывет"',
        'продавец слукавил. материал шарфв не является обещанным кашемиром.',
        'Браслет выглядит дорого, но застёжка слабая — расстегнулся через день, пришлось подклеить'
    ],
    'электроника': [
        'Наушники ужасные, звук глухой и один перестал работать через неделю.',
        'Клавиатура удобная, экран яркий, звук чистый. Всё как в описании.',
        'Кабель прочный, телефон заряжается быстро. Закажу ещё несколько.'
    ]
}

# стили для генерации разнообразных отзывов
styles = [
    'Сделай отзывы радостными, покупатель доволен, эмоции счастья и удовлетворения от товара',
    'Сделай отзывы критическими, покупатель разочарован, укажи недостатки товара или проблемы с размером, качеством, доставкой',
    'Отзывы нейтральные, описательные, без сильной эмоции, упор на детали товара, доставки и упаковки',
    'Добавь упоминание упаковки, цвета, размера, сроков доставки',
    'Очень краткие отзывы, 1–2 предложения, просто и понятно'
]

# собираем полную информацию для категорий
full_info = {}
for cat in categories:
    full_info[cat] = {'desc': description[cat], 'examples': examples[cat]}

{'бытовая техника': {'desc': 'холодильники, стиральные машины, плиты, микроволновки, чайники, пылесосы и другая техника для дома.', 'examples': ['Заказала стиральную машину, пришла быстро, упаковка целая, но при включении гремит и сильно прыгает по полу.', 'Пришёл с небольшой вмятиной, но работает без проблем. Доставка быстрая.', 'Плита греет хорошо, но запах пластика не проходил неделю. Сейчас всё нормально.']}, 'обувь': {'desc': 'кроссовки, ботинки, туфли, сандалии, сапоги и другая обувь для взрослых и детей.', 'examples': ['Купила новые кроссовки, очень удобные.', 'Качество отличное, запаха нет, фурнитура надёжная. Брала мужу — доволен.', 'Выглядят стильно, но ремешок короткий — на широкую ногу не подойдут.']}, 'посуда': {'desc': 'только тарелки, кружки, чашки для еды, столовые приборы и кухонная утварь; НЕ включает одежду, бельё или чашки бюстгальтеров.', 'examples': ['Тарелки пришли целые, хорошо упакованы. Очень красивые, рисунок яркий. Мыть можно в посудомойке — ничего не стерло

In [ ]:
# функция для составления промпта для генерации
def generation_prompt_builder(category_name, category_description, num_examples, seed_examples, style,
                              meta_info='''Отзывы в стиле маркетплейса, 1–3 коротких предложения, допустимы лёгкие орфографические ошибки. 
                              Старайся, чтобы каждый отзыв отличался формулировками, упоминай разные детали: 
                              запах, внешний вид, удобство, упаковку, срок службы, соответствие фото, цену.'''):
    prompt = f'''
    Сгенерируй {num_examples} реалистичных отзывов на русском языке
    Для категории {category_name}
    Описание категории: {category_description}
    {meta_info}
    {style}\n
    '''

    examples = '\n'.join([f'{i + 1}. {ex}' for i, ex in enumerate(seed_examples)])
    prompt += 'Вот примеры реальных отзывов:\n'
    prompt += examples + '\n'

    prompt += '\n\nОтвет верни строго в формате JSON-массива строк, например:\n["отзыв_1", "отзыв_2"]'

    return prompt

# получение ответа от llm
def get_llm_answer(prompt, max_tokens=512, temperature=0.0, top_p=1.0, do_sample=False):
    messages = [
        {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=max_tokens,
        do_sample=do_sample,
        temperature=temperature,
        top_p=top_p
    )
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 
    response = tokenizer.decode(output_ids, skip_special_tokens=True)

    return response

In [ ]:
# генерация данных
BATCH_SIZE = 5
result = {}
for cat in categories:
    result[cat] = []

for cat in full_info.keys():
    for i in tqdm(range(60), desc=f'Генерация для категории: {cat}', total=60):
        try:
            result[cat].append(get_llm_answer(generation_prompt_builder(
                cat,
                full_info[cat]['desc'],
                BATCH_SIZE,
                full_info[cat]['examples'],
                random.choice(styles),
            ), 4096, 0.7, 0.9, True))
        except Exception as e:
            print(f'Что то пошло не так: {e}')

### Генерация данных для проблемных ситуаций